In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd

#USGs_e_DNVs
caminho_planilha1 = '/content/drive/MyDrive/PRIVATE_DATA/dataframes_concatenados_22_08_todos_USGs_e_RNVs.xlsx'
caminho_planilha2 = '/content/drive/MyDrive/PRIVATE_DATA/RNVs_e_Adendos.xlsx'


df_abas = pd.read_excel(caminho_planilha1, sheet_name=None)
df_abas2 = pd.read_excel(caminho_planilha2, sheet_name=None)


# Acessar as abas pelo nome
concatenado = df_abas['Concatenado']
concatenado_exato = df_abas['Concatenado Exata']

# Acessar as abas pelo nome
rnv_com_adendos = df_abas2['Merged Final']
rnv_sem_adendos = df_abas2['DNV Sem Adendo']

In [ ]:
rnv_com_adendos.shape, rnv_sem_adendos.shape, concatenado.shape, concatenado_exato.shape

In [ ]:
# Exibir as colunas de concatenado e concatenado_exato para comparar
colunas_concatenado = concatenado.columns
colunas_concatenado_exato = concatenado_exato.columns

# Exibir as colunas de ambos os DataFrames
print("Colunas de 'concatenado':")
print(colunas_concatenado)

print("\nColunas de 'concatenado_exato':")
print(colunas_concatenado_exato)

# Converter as colunas para conjuntos para facilitar a comparação
colunas_concatenado = set(concatenado.columns)
colunas_concatenado_exato = set(concatenado_exato.columns)

# Colunas que estão em concatenado, mas não em concatenado_exato
diferenca_concatenado = colunas_concatenado - colunas_concatenado_exato

# Colunas que estão em concatenado_exato, mas não em concatenado
diferenca_concatenado_exato = colunas_concatenado_exato - colunas_concatenado

# Exibir as diferenças
print("Colunas em 'concatenado' que não estão em 'concatenado_exato':")
print(diferenca_concatenado)

print("\nColunas em 'concatenado_exato' que não estão em 'concatenado':")
print(diferenca_concatenado_exato)


In [ ]:
# Excluir as colunas de 'concatenado' que não estão em 'concatenado_exato'
colunas_para_excluir = ['Diferenças_Esperada_Dia', 'Idade_Gestacional_Exame', 'Idade_Gestacional_RNV']
concatenado = concatenado.drop(columns=colunas_para_excluir, errors='ignore')

# Adicionar a coluna 'Indice_Similaridade' em 'concatenado_exato' preenchida com valor 1
concatenado_exato['Indice_Similaridade'] = 1

# Exibir as primeiras linhas dos DataFrames modificados para verificar
print("Concatenado:")
print(concatenado.head())

print("\nConcatenado Exato:")
print(concatenado_exato.head())

Concatenando os dois dataframes os Concatenados e Concatenados Exatos e Inexatos

In [ ]:
# Verificar se as colunas estão na mesma ordem
colunas_concatenado = list(concatenado.columns)
colunas_concatenado_exato = list(concatenado_exato.columns)

# Comparar as colunas
if colunas_concatenado == colunas_concatenado_exato:
    print("As colunas estão na mesma ordem.")
else:
    print("As colunas NÃO estão na mesma ordem.")
    print("\nOrdem das colunas em 'concatenado':")
    print(colunas_concatenado)

    print("\nOrdem das colunas em 'concatenado_exato':")
    print(colunas_concatenado_exato)


In [ ]:
# Reordenar 'concatenado_exato' para ter a mesma ordem de colunas que 'concatenado'
concatenado_exato = concatenado_exato[concatenado.columns]

# Verificar se agora as colunas estão na mesma ordem
if list(concatenado.columns) == list(concatenado_exato.columns):
    print("As colunas estão agora na mesma ordem e prontas para a concatenação.")
else:
    print("As colunas ainda não estão na mesma ordem.")


In [ ]:
# Concatenar os DataFrames, colocando 'concatenado_exato' antes de 'concatenado'
df_concatenado_rnv_usg = pd.concat([concatenado_exato, concatenado], ignore_index=True)

concatenado_exato.shape,concatenado.shape,df_concatenado_rnv_usg.shape

In [ ]:
# Mesclar rnv_sem_adendos com concatenado baseado na coluna 'NºDNV'
merged_concatenado = pd.merge(rnv_sem_adendos, df_concatenado_rnv_usg, on='NºDNV', how='left')

merged_concatenado_com_adendos = pd.merge(rnv_com_adendos, df_concatenado_rnv_usg, on='NºDNV', how='left')

print(merged_concatenado.shape,merged_concatenado_com_adendos.shape)
print(rnv_sem_adendos.shape,rnv_com_adendos.shape)


In [ ]:
# Salvando os DataFrames 'merged_concatenado' e 'merged_concatenado_com_adendos' em um arquivo Excel
with pd.ExcelWriter('01_10_2024.xlsx') as writer:
    merged_concatenado.to_excel(writer, sheet_name='RNV e USGs', index=False)
    merged_concatenado_com_adendos.to_excel(writer, sheet_name='RNV,USG e Adendos', index=False)

# Exibir mensagem de confirmação
print("Os DataFrames foram salvos no arquivo 'merged_dataframes_result.xlsx'.")

##Daqui pra Frente é pra fazer as atualizações do Renato Acordado em 02/10

In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
merged_concatenado_com_adendos['Texto_x']

In [ ]:
#Coluna PIG

merged_concatenado_com_adendos['PIG'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: 1 if any(phrase in x for phrase in ['PIG', 'PequenopraIdadeGestacional','pequenopraidadegestacional', 'P05', 'P.05', 'P-05']) else 0
)

# Exibir o DataFrame atualizado
merged_concatenado_com_adendos[['Texto_x', 'PIG']]

In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
#Baixo Peso

merged_concatenado_com_adendos['Baixo Peso'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: 1 if any(phrase in x for phrase in ['baixopeso', 'BaixoPeso', 'Baixopeso', 'baixoPeso', 'BP', 'BPN', 'MBP', 'P07', 'P.07', 'P-07']) else 0
)

merged_concatenado_com_adendos[['Texto_x', 'Baixo Peso']]

In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
#Asfixia

merged_concatenado_com_adendos['Asfixia'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: 1 if any(phrase in x for phrase in ['Asfixia', 'asfixia', 'P21', 'P-21', 'P.21', 'acidosemetabólica',
                                                  'acidosemetabolica', 'encefalopatiahipoxico-isquêmica', 'EHI',
                                                  'P91', 'P.91', 'P-91']) else 0
)

merged_concatenado_com_adendos[['Texto_x', 'Asfixia']]

In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
import re

# Função que procura as expressões exatamente no texto
def check_exact_match(text, patterns):
    return 1 if any(re.search(r'\b' + pattern + r'\b', text) for pattern in patterns) else 0

# Criar a nova coluna 'Enterocolite necrosante' com base nas expressões fornecidas
patterns = ['EnterocoliteNecrosante', 'enterocolitenecrosante', 'enterocoliteNecosante', 'EN', 'P77', 'P.77', 'P-77']

merged_concatenado_com_adendos['Enterocolite necrosante'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)


merged_concatenado_com_adendos[['Texto_x', 'Enterocolite necrosante']]

In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
patterns = ['LeucomaláciaPeriventricular', 'leucomaláciaperiventricular', 'LeucomalaciaPeriventricular', 'leucomalaciaperiventricular',
            'LPV', 'P91', 'P.91', 'P-91']

merged_concatenado_com_adendos['Leucomalácia periventricular'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

frequencia_leucomalacia = merged_concatenado_com_adendos['Leucomalácia periventricular'].value_counts(normalize=True)

frequencia_leucomalacia


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
# Criar a nova coluna 'Crise convulsiva' com base nas expressões fornecidas
patterns = ['CriseConvulsiva', 'criseconvulsiva', 'Convulsão', 'convulsão', 'ConvulsãoNeonatal', 'convulsãoneonatal',
            'P90', 'P.90', 'P-90']

merged_concatenado_com_adendos['Crise convulsiva'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

# Calcular a frequência de 0 e 1 na nova coluna
frequencia_crise_convulsiva = merged_concatenado_com_adendos['Crise convulsiva'].value_counts(normalize=True)

# Exibir a frequência
frequencia_crise_convulsiva

In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:

def check_hemorragia_intraventricular(text):
    patterns = ['HemorragiaIntraventricular', 'hemorragiaintraventricular',
                'HPIV', 'HPiv', 'Hiv', 'HIV', 'hemorragiaPIV', 'HemorragiaPeriintraventricular',
                'hemorragiaPIV', 'hemorragiaperiintraventricular', 'P52', 'P.52', 'P-52']

    if re.search(r'HIV\s*(neg|Neg|NEG|Negativo|negativo|NEGATIVO)', text):
        return 0

    return 1 if any(re.search(r'\b' + pattern + r'\b', text) for pattern in patterns) else 0

merged_concatenado_com_adendos['Hemorragia intraventricular'] = merged_concatenado_com_adendos['Texto_x'].apply(check_hemorragia_intraventricular)

frequencia_hemorragia_intraventricular = merged_concatenado_com_adendos['Hemorragia intraventricular'].value_counts(normalize=True)

frequencia_hemorragia_intraventricular


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
patterns = ['DisplasiaBroncopulmonar', 'displasiabroncopulmonar', 'DBP', 'P27', 'P.27', 'P-27']

merged_concatenado_com_adendos['Displasia broncopulmonar'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

frequencia_displasia_broncopulmonar = merged_concatenado_com_adendos['Displasia broncopulmonar'].value_counts(normalize=True)

frequencia_displasia_broncopulmonar


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
# Criar a nova coluna 'Desconforto respiratório' com base nas expressões fornecidas
patterns = ['DesconfortoRespiratório', 'desconfortorespiratório', 'SíndromedoDesconfortoRespiratório',
            'síndromedodesconfortorespiratório', 'SDR', 'DRP', 'desconfortorespiratórioprecoce',
            'DesconfortoRespiratórioPrecoce', 'P22', 'P.22', 'P-22', 'P22.8', 'P.22.8', 'P-22.8']

merged_concatenado_com_adendos['Desconforto respiratório'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

# Calcular a frequência de 0 e 1 na nova coluna
frequencia_desconforto_respiratorio = merged_concatenado_com_adendos['Desconforto respiratório'].value_counts(normalize=True)

# Exibir a frequência
frequencia_desconforto_respiratorio


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
patterns = ['NecessidadedeVentilaçãoMecânica', 'necessidadedeventilaçãomecânica',
            'NecessidadedeVentilacaoMecanica', 'necessidadedeventilacaomecanica',
            'SuporteVentilatório', 'suporteventilatório',
            'SuporteVentilatorio', 'suporteventilatorio',
            'intubação', 'Intubação',
            'intubacao', 'Intubacao']
merged_concatenado_com_adendos['Necessidade de ventilação mecânica'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

# Calcular a frequência de 0 e 1 na nova coluna
frequencia_ventilacao_mecanica = merged_concatenado_com_adendos['Necessidade de ventilação mecânica'].value_counts(normalize=True)

# Exibir a frequência
frequencia_ventilacao_mecanica


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
# Criar a nova coluna 'Hipoglicemia neonatal' com base nas expressões fornecidas
patterns = ['HipoglicemiaNeonatal', 'hipoglicemianeonatal',
            'P70.4', 'P-70.4', 'P70', 'P.70', 'P-70']

merged_concatenado_com_adendos['Hipoglicemia neonatal'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

# Calcular a frequência de 0 e 1 na nova coluna
frequencia_hipoglicemia_neonatal = merged_concatenado_com_adendos['Hipoglicemia neonatal'].value_counts(normalize=True)

# Exibir a frequência
frequencia_hipoglicemia_neonatal


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
# Criar a nova coluna 'Icterícia neonatal' com base nas expressões fornecidas
patterns = ['IcteríciaNeonatal', 'icterícianeonatal',
            'IctericiaNeonatal', 'ictericianeonatal',  # sem acentuação
            'INN', 'P59', 'P.59', 'P-59', 'P59.9', 'P.59.9', 'P-59.9']

merged_concatenado_com_adendos['Icterícia neonatal'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

# Calcular a frequência de 0 e 1 na nova coluna
frequencia_ictericia_neonatal = merged_concatenado_com_adendos['Icterícia neonatal'].value_counts(normalize=True)

# Exibir a frequência
frequencia_ictericia_neonatal


In [ ]:
# Criar a nova coluna 'Recém nascido pré termo' com base nas expressões fornecidas
patterns = ['RecémNascidoPréTermo', 'recémnascidoprétermo',
            'RecemNascidoPreTermo', 'recemnascidopretermo', 'RecémNascidoPré-Termo', 'recémnascidopré-termo',
            'RecemNascidoPre-Termo', 'recemnascidopre-termo',  # sem acentuação
            'RNPT', 'P07', 'P.07', 'P-07']

merged_concatenado_com_adendos['Recém nascido pré termo'] = merged_concatenado_com_adendos['Texto_x'].apply(
    lambda x: check_exact_match(x, patterns)
)

# Calcular a frequência de 0 e 1 na nova coluna
frequencia_recem_nascido_pre_termo = merged_concatenado_com_adendos['Recém nascido pré termo'].value_counts(normalize=True)

# Exibir a frequência
frequencia_recem_nascido_pre_termo


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
# Definir o intervalo de semanas
semanas = range(4, 43)

# Iterar sobre cada semana para criar as novas colunas
for semana in semanas:
    merged_concatenado[f'Texto_{semana}'] = merged_concatenado['Texto'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Data do Exame_{semana}'] = merged_concatenado['Data do Exame'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Data de Nascimento_Mãe_{semana}'] = merged_concatenado['Data de Nascimento_Mãe'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Idade Gestacional Dias_{semana}'] = merged_concatenado['Idade Gestacional Dias'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Frequencia Cardiaca_{semana}'] = merged_concatenado['Frequencia Cardiaca'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Peso Fetal_{semana}'] = merged_concatenado['Peso Fetal'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Diametro Biparietal_{semana}'] = merged_concatenado['Diametro Biparietal'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Circunferência Cefálica_{semana}'] = merged_concatenado['Circunferência Cefálica'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Comprimento Femural_{semana}'] = merged_concatenado['Comprimento Femural'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Circunferência Abdominal_{semana}'] = merged_concatenado['Circunferência Abdominal'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'TIPO_DE_ULTRASSOM_{semana}'] = merged_concatenado['TIPO_DE_ULTRASSOM'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'Ducto_Venoso_{semana}'] = merged_concatenado['Ducto_Venoso'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado[f'DopplerFluxometria_{semana}'] = merged_concatenado['DopplerFluxometria'].where(merged_concatenado['Idade Gestacional Semanas'] == semana, None)

# Verifique o DataFrame resultante
merged_concatenado.head()


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
merged_concatenado.head()


In [ ]:
# Definir o intervalo de semanas
semanas = range(4, 43)

# Iterar sobre cada semana para criar as novas colunas em 'merged_concatenado_com_adendos'
for semana in semanas:
    merged_concatenado_com_adendos[f'Texto_{semana}'] = merged_concatenado_com_adendos['Texto_x'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Data do Exame_{semana}'] = merged_concatenado_com_adendos['Data do Exame'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Data de Nascimento_Mãe_{semana}'] = merged_concatenado_com_adendos['Data de Nascimento_Mãe'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Idade Gestacional Dias_{semana}'] = merged_concatenado_com_adendos['Idade Gestacional Dias'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Frequencia Cardiaca_{semana}'] = merged_concatenado_com_adendos['Frequencia Cardiaca'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Peso Fetal_{semana}'] = merged_concatenado_com_adendos['Peso Fetal'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Diametro Biparietal_{semana}'] = merged_concatenado_com_adendos['Diametro Biparietal'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Circunferência Cefálica_{semana}'] = merged_concatenado_com_adendos['Circunferência Cefálica'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Comprimento Femural_{semana}'] = merged_concatenado_com_adendos['Comprimento Femural'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Circunferência Abdominal_{semana}'] = merged_concatenado_com_adendos['Circunferência Abdominal'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'TIPO_DE_ULTRASSOM_{semana}'] = merged_concatenado_com_adendos['TIPO_DE_ULTRASSOM'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'Ducto_Venoso_{semana}'] = merged_concatenado_com_adendos['Ducto_Venoso'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)
    merged_concatenado_com_adendos[f'DopplerFluxometria_{semana}'] = merged_concatenado_com_adendos['DopplerFluxometria'].where(merged_concatenado_com_adendos['Idade Gestacional Semanas'] == semana, None)

# Verifique o DataFrame resultante
merged_concatenado_com_adendos.head()


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
# Excluir as colunas base de 'merged_concatenado_com_adendos', mantendo apenas as colunas com semanas
colunas_base_adendos = ['Texto_x', 'Data do Exame', 'Data de Nascimento_Mãe', 'Idade Gestacional Semanas',
                        'Idade Gestacional Dias', 'Frequencia Cardiaca', 'Peso Fetal', 'Diametro Biparietal',
                        'Circunferência Cefálica', 'Comprimento Femural', 'Circunferência Abdominal',
                        'TIPO_DE_ULTRASSOM', 'Ducto_Venoso', 'DopplerFluxometria']
merged_concatenado_com_adendos = merged_concatenado_com_adendos.drop(columns=colunas_base_adendos)

# Excluir as colunas base de 'merged_concatenado', mantendo apenas as colunas com semanas
colunas_base_concatenado = ['Texto', 'Data do Exame', 'Data de Nascimento_Mãe', 'Idade Gestacional Semanas',
                            'Idade Gestacional Dias', 'Frequencia Cardiaca', 'Peso Fetal', 'Diametro Biparietal',
                            'Circunferência Cefálica', 'Comprimento Femural', 'Circunferência Abdominal',
                            'TIPO_DE_ULTRASSOM', 'Ducto_Venoso', 'DopplerFluxometria']
merged_concatenado = merged_concatenado.drop(columns=colunas_base_concatenado)

# Verificar os DataFrames resultantes
merged_concatenado_com_adendos.head()
merged_concatenado.head()


In [ ]:
merged_concatenado_com_adendos.shape, merged_concatenado.shape

In [ ]:
# Função para agrupar as linhas com base no 'NºDNV'
def agrupar_linhas_por_dnvs(df, coluna_identificacao):
    # Inicializar uma lista para armazenar o resultado
    linhas_agrupadas = []

    # Iterar por grupo de 'NºDNV'
    for dn, group in df.groupby(coluna_identificacao):
        # Inicializar um dicionário para armazenar os dados da linha agrupada
        linha_agrupada = {}

        # Iterar pelas colunas do grupo
        for col in group.columns:
            # Se todas as linhas da coluna forem iguais (desconsiderando NaNs), mantenha a informação
            if group[col].nunique(dropna=True) == 1:
                linha_agrupada[col] = group[col].dropna().iloc[0] if not group[col].isna().all() else None
            # Caso contrário, concatene as informações diferentes, incluindo os valores válidos
            else:
                linha_agrupada[col] = ' | '.join(group[col].dropna().astype(str).unique())

        # Adicionar a linha agrupada à lista de resultados
        linhas_agrupadas.append(linha_agrupada)

    # Converter a lista de dicionários de volta para um DataFrame
    df_agrupado = pd.DataFrame(linhas_agrupadas)

    return df_agrupado

# Aplicar a função aos DataFrames
merged_concatenado_com_adendos_agrupado = agrupar_linhas_por_dnvs(merged_concatenado_com_adendos, 'NºDNV')
merged_concatenado_agrupado = agrupar_linhas_por_dnvs(merged_concatenado, 'NºDNV')


In [ ]:
merged_concatenado_com_adendos_agrupado

In [ ]:
pd.set_option('display.max_columns', None)  # Mostra todas as colunas
pd.set_option('display.max_rows', 10)  # Limita o número de linhas exibidas para 10
merged_concatenado_com_adendos

In [ ]:
# Salvando os DataFrames 'merged_concatenado' e 'merged_concatenado_com_adendos' em um arquivo Excel
with pd.ExcelWriter('07_10_2024_5.xlsx') as writer:
    merged_concatenado_agrupado.to_excel(writer, sheet_name='RNV e USGs', index=False)
    merged_concatenado_com_adendos_agrupado.to_excel(writer, sheet_name='RNV,USG e Adendos', index=False)

# Exibir mensagem de confirmação
print("Os DataFrames foram salvos no arquivo 'merged_dataframes_result.xlsx'.")

##Juntando em Um Mesmo Arquivo conforme solicitação do Renato

Como fiz alguns ajustes manuais vou puxar denovo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Caminho para o arquivo no Google Drive
file_path = '/content/drive/MyDrive/PRIVATE_DATA/07_10_2024_5.xlsx'

# Ler as duas abas do arquivo Excel
merged_concatenado_agrupado = pd.read_excel(file_path, sheet_name=0)
merged_concatenado_com_adendos_agrupado = pd.read_excel(file_path, sheet_name=1)

# Verificar as primeiras linhas de cada DataFrame (opcional)
print(merged_concatenado_agrupado.head())
print(merged_concatenado_com_adendos_agrupado.head())


In [ ]:
# Verificar as colunas de cada DataFrame
colunas_agrupado = set(merged_concatenado_agrupado.columns)
colunas_com_adendos_agrupado = set(merged_concatenado_com_adendos_agrupado.columns)

# Identificar colunas que estão em ambos os DataFrames
colunas_comuns = colunas_agrupado.intersection(colunas_com_adendos_agrupado)
print("Colunas comuns:", colunas_comuns)

# Identificar colunas que estão apenas em cada DataFrame
colunas_somente_agrupado = colunas_agrupado - colunas_com_adendos_agrupado
colunas_somente_com_adendos_agrupado = colunas_com_adendos_agrupado - colunas_agrupado

print("Colunas apenas em 'merged_concatenado_agrupado':", colunas_somente_agrupado)
print("Colunas apenas em 'merged_concatenado_com_adendos_agrupado':", colunas_somente_com_adendos_agrupado)


In [ ]:
# Remover a coluna 'Indice_Similaridade' de merged_concatenado_agrupado, se existir
if 'Indice_Similaridade' in merged_concatenado_agrupado.columns:
    merged_concatenado_agrupado = merged_concatenado_agrupado.drop(columns=['Indice_Similaridade'])

# Lista de colunas a serem acrescentadas em merged_concatenado_agrupado
novas_colunas = {
    'Sexo', 'Hipoglicemia neonatal', 'Quantidade Consultas Pré-Natal', 'PC', 'Texto_y',
    'Distancia de Levenshtein', 'Desconforto respiratório', 'Tipo de Sangue',
    'Displasia broncopulmonar', 'Nome da Mãe', 'Hemorragia intraventricular', 'Baixo Peso',
    'Leucomalácia periventricular', 'Necessidade de ventilação mecânica', 'Peso', 'PT',
    'Icterícia neonatal', 'Nome Exato', 'Crise convulsiva', 'Nome do Arquivo', 'Asfixia',
    'Dt. Óbito', 'PIG', 'Recém nascido pré termo', 'Comprimento', 'Informação Relevante',
    'Enterocolite necrosante', 'Data de Nascimento', 'Tipo de Parto'
}

# Adicionar as novas colunas, preenchendo com NaN inicialmente
for coluna in novas_colunas:
    if coluna not in merged_concatenado_agrupado.columns:
        merged_concatenado_agrupado[coluna] = pd.NA

# Verificar as primeiras linhas para confirmar as alterações (opcional)
print(merged_concatenado_agrupado.head())


In [ ]:
# Ajustar a ordem das colunas de merged_concatenado_agrupado
colunas_ordem = merged_concatenado_com_adendos_agrupado.columns

# Reordenar as colunas de merged_concatenado_agrupado para que sigam a mesma ordem
merged_concatenado_agrupado = merged_concatenado_agrupado.reindex(columns=colunas_ordem)

# Verificar as primeiras linhas para confirmar as alterações (opcional)
print(merged_concatenado_agrupado.head())

In [ ]:
# Adicionar a coluna 'Internação' em merged_concatenado_agrupado com valor "Não"
merged_concatenado_agrupado['Internação'] = 'Não'

# Adicionar a coluna 'Internação' em merged_concatenado_com_adendos_agrupado com valor "Sim"
merged_concatenado_com_adendos_agrupado['Internação'] = 'Sim'

# Verificar as primeiras linhas para confirmar as alterações (opcional)
print(merged_concatenado_agrupado[['Internação']].head())
print(merged_concatenado_com_adendos_agrupado[['Internação']].head())

In [ ]:
# Concatenar os DataFrames ao longo das linhas
merged_final = pd.concat([merged_concatenado_agrupado, merged_concatenado_com_adendos_agrupado], ignore_index=True)

# Verificar as primeiras linhas do DataFrame concatenado (opcional)
print(merged_final.head())


In [ ]:
merged_final_columns = merged_final.columns.tolist()

# Exibir todas as colunas
print(merged_final_columns)

In [ ]:

# Filtrar os casos onde 'MUNICIPIO_x' corresponde a qualquer variação de 'JUNDIAI'
filtro_jundiai = merged_final['MUNICIPIO_x'].str.lower().str.replace('í', 'i', regex=False).str.replace('ì', 'i', regex=False) == 'jundiai'

# Aplicar o filtro ao DataFrame
merged_final_filtered = merged_final[filtro_jundiai]

# Verificar as primeiras linhas do DataFrame filtrado (opcional)
print(merged_final_filtered.head())


In [ ]:
# Calcular a frequência dos valores na coluna 'MUNICIPIO_x' em merged_final
frequencia_municipio_x = merged_final['MUNICIPIO_x'].value_counts()
pd.set_option('display.max_rows', None)

frequencia_municipio_x

In [ ]:
# Filtrar as linhas que possuem as variações desejadas na coluna 'MUNICIPIO_x'
variations = ['JUNDIAI', 'JUNDIAÍ', 'JUINDIAÍ', 'JUNDIA', 'JUNDIAí', 'JNDIAI']
merged_final_filtered_df = merged_final[merged_final['MUNICIPIO_x'].isin(variations)]

# O DataFrame resultante manterá todas as colunas de 'merged_final'
merged_final_filtered_df


In [ ]:
# Calcular a frequência dos valores na coluna 'GEMELAR'
frequencia_gemelar = merged_final_filtered_df['GEMELAR'].value_counts()

# Exibir a frequência
print(frequencia_gemelar)

In [ ]:
numero_dnv_unicos = merged_final_filtered_df['NºDNV'].nunique()
numero_dnv_unicos

In [ ]:
merged_final_filtered_df = merged_final_filtered_df[merged_final_filtered_df['GEMELAR'] == 0]

In [ ]:
merged_final_filtered_df

In [ ]:
merged_final_filtered_df.to_excel('merged_final_filtered_df.xlsx', index=False)